In [ ]:
# ============================================================
# GREEK TEXT DATASET - NLP + K-MEANS CLUSTERING
# Dataset: greek.csv
# ============================================================

# ------------------------------------------------------------
# 0. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------
# Run this cell if the libraries are not already installed.

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import nltk
from nltk.corpus import stopwords


# ------------------------------------------------------------
# 2. DOWNLOAD NLTK RESOURCES
# ------------------------------------------------------------

nltk.download("stopwords")


# ------------------------------------------------------------
# 3. LOAD THE CSV DATASET
# ------------------------------------------------------------

FILE_NAME = "greek.csv"

df = pd.read_csv(FILE_NAME)

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ------------------------------------------------------------
# 4. DISPLAY DATASET INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 5. FIND TEXT COLUMN
# ------------------------------------------------------------

# Find columns containing text
text_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nText columns found:")
print(text_columns)

if len(text_columns) == 0:
    raise ValueError(
        "No text column was found in greek.csv."
    )

# Automatically select the first text column
TEXT_COLUMN = text_columns[0]

print("\nText column selected:")
print(TEXT_COLUMN)


# ------------------------------------------------------------
# 6. CLEAN THE DATASET
# ------------------------------------------------------------

# Remove rows with missing text
df = df.dropna(
    subset=[TEXT_COLUMN]
).copy()

# Convert text to string
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

# Remove duplicate documents
df = df.drop_duplicates(
    subset=[TEXT_COLUMN]
).copy()

# Reset index
df = df.reset_index(drop=True)

print("\nDataset shape after cleaning:")
print(df.shape)


# ------------------------------------------------------------
# 7. GREEK STOPWORDS
# ------------------------------------------------------------

greek_stopwords = set(
    stopwords.words("greek")
)

print("\nNumber of Greek stopwords:")
print(len(greek_stopwords))

print("\nSome Greek stopwords:")
print(list(greek_stopwords)[:30])


# ------------------------------------------------------------
# 8. GREEK TEXT PREPROCESSING FUNCTION
# ------------------------------------------------------------

def clean_greek_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Keep Greek characters, Greek accented characters,
    # numbers and spaces
    text = re.sub(
        r"[^α-ωάέήίόύώϊϋΐΰ0-9\s]",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\b\d+\b",
        " ",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Split into words
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in greek_stopwords
    ]

    # Remove very short words
    words = [
        word
        for word in words
        if len(word) > 2
    ]

    # Return cleaned text
    return " ".join(words)


# ------------------------------------------------------------
# 9. APPLY NLP PREPROCESSING
# ------------------------------------------------------------

df["clean_text"] = df[TEXT_COLUMN].apply(
    clean_greek_text
)

print("\nOriginal and cleaned text:")
display(
    df[
        [TEXT_COLUMN, "clean_text"]
    ].head(10)
)


# ------------------------------------------------------------
# 10. REMOVE EMPTY DOCUMENTS
# ------------------------------------------------------------

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

print("\nNumber of documents after NLP cleaning:")
print(len(df))


# ------------------------------------------------------------
# 11. TF-IDF VECTORIZATION
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print("\n" + "=" * 70)
print("TF-IDF INFORMATION")
print("=" * 70)

print("\nTF-IDF matrix shape:")
print(X.shape)

print("\nNumber of documents:")
print(X.shape[0])

print("\nNumber of features:")
print(X.shape[1])


# ------------------------------------------------------------
# 12. DISPLAY TF-IDF FEATURES
# ------------------------------------------------------------

feature_names = vectorizer.get_feature_names_out()

print("\nFirst 50 TF-IDF features:")
print(feature_names[:50])


# ------------------------------------------------------------
# 13. FIND OPTIMAL NUMBER OF CLUSTERS
# ------------------------------------------------------------

# We will test K values from 2 to 10.
# If the dataset is very small, adjust automatically.

max_k = min(
    10,
    len(df) - 1
)

if max_k < 2:
    raise ValueError(
        "The dataset needs at least 3 documents for K-Means clustering."
    )

k_values = list(
    range(2, max_k + 1)
)

silhouette_scores = []
inertias = []


print("\n" + "=" * 70)
print("TESTING DIFFERENT VALUES OF K")
print("=" * 70)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
        max_iter=300
    )

    labels = model.fit_predict(X)

    # Inertia
    inertias.append(
        model.inertia_
    )

    # Silhouette score
    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(
        score
    )

    print(
        f"K = {k:2d} | "
        f"Inertia = {model.inertia_:.4f} | "
        f"Silhouette = {score:.4f}"
    )


# ------------------------------------------------------------
# 14. ELBOW METHOD
# ------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    k_values,
    inertias,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xticks(k_values)

plt.grid(True)

plt.show()


# ------------------------------------------------------------
# 15. SILHOUETTE SCORE PLOT
# ------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    k_values,
    silhouette_scores,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(k_values)

plt.grid(True)

plt.show()


# ------------------------------------------------------------
# 16. SELECT BEST K
# ------------------------------------------------------------

best_index = np.argmax(
    silhouette_scores
)

best_k = k_values[
    best_index
]

best_score = silhouette_scores[
    best_index
]

print("\n" + "=" * 70)
print("BEST K")
print("=" * 70)

print(
    "Best number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(best_score, 4)
)


# ------------------------------------------------------------
# 17. TRAIN FINAL K-MEANS MODEL
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10,
    max_iter=300
)

cluster_labels = kmeans.fit_predict(
    X
)

# Add cluster to dataframe
df["cluster"] = cluster_labels


# ------------------------------------------------------------
# 18. CLUSTER DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLUSTER DISTRIBUTION")
print("=" * 70)

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(cluster_counts)


# ------------------------------------------------------------
# 19. PLOT CLUSTER DISTRIBUTION
# ------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Documents"
)

plt.title(
    "Number of Documents per Cluster"
)

plt.show()


# ------------------------------------------------------------
# 20. FIND IMPORTANT WORDS FOR EACH CLUSTER
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP WORDS FOR EACH CLUSTER")
print("=" * 70)

terms = vectorizer.get_feature_names_out()

order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)

TOP_N_WORDS = 20

cluster_keywords = {}

for cluster_number in range(best_k):

    top_words = [
        terms[index]
        for index in order_centroids[
            cluster_number,
            :TOP_N_WORDS
        ]
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ------------------------------------------------------------
# 21. DISPLAY KEYWORDS AS A DATAFRAME
# ------------------------------------------------------------

keywords_df = pd.DataFrame(
    dict(
        (
            cluster,
            pd.Series(words)
        )
        for cluster, words
        in cluster_keywords.items()
    )
)

keywords_df.index = [
    f"Word {i + 1}"
    for i in range(TOP_N_WORDS)
]

print("\nKeywords by cluster:")
display(keywords_df)


# ------------------------------------------------------------
# 22. DISPLAY DOCUMENTS IN EACH CLUSTER
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE DOCUMENTS FROM EACH CLUSTER")
print("=" * 70)

SAMPLES_PER_CLUSTER = 5

for cluster_number in range(best_k):

    print("\n")
    print("=" * 70)
    print(
        f"CLUSTER {cluster_number}"
    )
    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    samples = cluster_data[
        TEXT_COLUMN
    ].head(
        SAMPLES_PER_CLUSTER
    )

    for i, text in enumerate(
        samples,
        start=1
    ):
        print(
            f"\n{i}. {text}"
        )


# ------------------------------------------------------------
# 23. PCA VISUALIZATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PCA VISUALIZATION")
print("=" * 70)

# PCA requires dense data
X_dense = X.toarray()

# Reduce to two dimensions
pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

print(
    "Explained variance:",
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    round(
        pca.explained_variance_ratio_.sum(),
        4
    )
)


# ------------------------------------------------------------
# 24. PLOT K-MEANS CLUSTERS
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    cmap="viridis",
    alpha=0.7,
    s=50
)

plt.xlabel(
    "PCA Component 1"
)

plt.ylabel(
    "PCA Component 2"
)

plt.title(
    "Greek Text - K-Means Clustering"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ------------------------------------------------------------
# 25. CALCULATE FINAL SILHOUETTE SCORE
# ------------------------------------------------------------

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette Score:",
    round(
        final_silhouette,
        4
    )
)

print(
    "K-Means Inertia:",
    round(
        kmeans.inertia_,
        4
    )
)


# ------------------------------------------------------------
# 26. SHOW DOCUMENT + CLUSTER
# ------------------------------------------------------------

result_columns = [
    TEXT_COLUMN,
    "clean_text",
    "cluster"
]

print("\n" + "=" * 70)
print("FINAL CLUSTERED DATA")
print("=" * 70)

display(
    df[result_columns].head(20)
)


# ------------------------------------------------------------
# 27. SORT DOCUMENTS BY CLUSTER
# ------------------------------------------------------------

df_sorted = df.sort_values(
    by="cluster"
).reset_index(
    drop=True
)

display(
    df_sorted[
        result_columns
    ].head(20)
)


# ------------------------------------------------------------
# 28. SAVE RESULTS TO CSV
# ------------------------------------------------------------

OUTPUT_FILE = (
    "greek_kmeans_results.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"\nResults saved successfully to: "
    f"{OUTPUT_FILE}"
)


# ------------------------------------------------------------
# 29. SAVE CLUSTER KEYWORDS
# ------------------------------------------------------------

keywords_output = []

for cluster_number in range(best_k):

    for rank, word in enumerate(
        cluster_keywords[cluster_number],
        start=1
    ):

        keywords_output.append({
            "cluster": cluster_number,
            "rank": rank,
            "keyword": word
        })

keywords_output_df = pd.DataFrame(
    keywords_output
)

keywords_output_df.to_csv(
    "greek_cluster_keywords.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Cluster keywords saved to: "
    "greek_cluster_keywords.csv"
)


# ------------------------------------------------------------
# 30. FINAL SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

print(
    f"Original dataset: {FILE_NAME}"
)

print(
    f"Text column: {TEXT_COLUMN}"
)

print(
    f"Number of documents: {len(df)}"
)

print(
    f"Number of TF-IDF features: {X.shape[1]}"
)

print(
    f"Best K: {best_k}"
)

print(
    f"Silhouette Score: "
    f"{final_silhouette:.4f}"
)

print(
    "\nOutput files:"
)

print(
    "1. greek_kmeans_results.csv"
)

print(
    "2. greek_cluster_keywords.csv"
)